# 07 Hybrid Search and Reranking

## Goal

This notebook improves retrieval quality for RiskRadar AI.

The workflow is:

```text
user question
→ vector search
→ keyword search
→ merge results
→ rerank evidence
→ return stronger citation-ready chunks
```

Notebook 06 used semantic vector search only.

This notebook adds:

```text
BM25 keyword search
+ reciprocal rank fusion
+ cross-encoder reranking
```

Why this matters:

Vector search is good at meaning.

Keyword search is good at exact terms.

Reranking helps reorder the best candidates before answer generation.

This gives the RAG system a stronger evidence layer.

## Package Note

This notebook uses `rank-bm25` for BM25 keyword retrieval.

BM25 helps the RAG system find exact keyword matches such as:

```text
cybersecurity
supply chain
competition
regulation
artificial intelligence
```

This complements vector search, which is better at finding semantic meaning.

In [1]:
# Import tools for file and folder paths
from pathlib import Path

# Import pandas for working with tables
import pandas as pd

# Import numpy for numerical work
import numpy as np

# Import regular expressions for text cleaning and tokenization
import re

# Import ChromaDB for vector search
import chromadb

# Import SentenceTransformer for query embeddings
from sentence_transformers import SentenceTransformer

# Import CrossEncoder for reranking retrieved chunks
from sentence_transformers import CrossEncoder

# Import BM25 keyword search
from rank_bm25 import BM25Okapi

# Import textwrap for cleaner text previews
import textwrap

# Import time for timing retrieval methods
import time

In [2]:
# Detect the project root automatically
# If this notebook is inside the notebooks folder, move one level up
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Create main project paths
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
VECTORSTORE_DIR = DATA_DIR / "vectorstore"

# Set Chroma vector database folder
CHROMA_DIR = VECTORSTORE_DIR / "chroma_sec_10k"

# Print paths to confirm everything is correct
print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)
print("Vectorstore folder:", VECTORSTORE_DIR)
print("Chroma folder:", CHROMA_DIR)

Project root: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI
Processed data folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed
Vectorstore folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\vectorstore
Chroma folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\vectorstore\chroma_sec_10k


In [3]:
# Set path to final RAG chunks from notebook 04
rag_chunks_file = PROCESSED_DIR / "sec_10k_rag_chunks.csv"

# Check that the chunk file exists
if not rag_chunks_file.exists():
    raise FileNotFoundError(
        f"Could not find {rag_chunks_file}. Run 04_chunking_experiments.ipynb first."
    )

# Load RAG chunks
rag_chunks_df = pd.read_csv(rag_chunks_file)

# Fill missing values so search metadata does not break
rag_chunks_df = rag_chunks_df.fillna("")

# Preview chunk dataset
rag_chunks_df.head()

,chunk_id,document_id,ticker,company_name,filing_date,accession_number,filing_url,section_name,source_label,citation_label,chunk_index,chunk_size,chunk_overlap,start_word,end_word,chunk_word_count,chunk_character_count,chunk_text
0,AAPL_2025-10-31_item_1_business_chunk_0000,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 0",0,350,75,0,350,350,2125,Item 1. Business Company Background The Compan...
1,AAPL_2025-10-31_item_1_business_chunk_0001,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 1",1,350,75,275,625,350,2341,Apple Inc. | 2025 Form 10-K | 1 Services Adver...
2,AAPL_2025-10-31_item_1_business_chunk_0002,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 2",2,350,75,550,900,350,2429,"Greater China includes China mainland, Hong Ko..."
3,AAPL_2025-10-31_item_1_business_chunk_0003,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 3",3,350,75,825,1175,350,2563,by imitating the Company’s products and infrin...
4,AAPL_2025-10-31_item_1_business_chunk_0004,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 4",4,350,75,1100,1450,350,2357,provide products and services at little or no ...


#### Load embedding summary

In [4]:
# Set path to embedding summary from notebook 05
embedding_summary_file = PROCESSED_DIR / "sec_10k_embedding_summary.csv"

# Check that the embedding summary exists
if not embedding_summary_file.exists():
    raise FileNotFoundError(
        f"Could not find {embedding_summary_file}. Run 05_embeddings_and_vector_store.ipynb first."
    )

# Load embedding summary
embedding_summary = pd.read_csv(embedding_summary_file)

# Convert embedding summary into a dictionary
embedding_settings = dict(
    zip(
        embedding_summary["setting"],
        embedding_summary["value"]
    )
)

# Pull settings created in notebook 05
EMBEDDING_MODEL_NAME = embedding_settings["embedding_model"]
COLLECTION_NAME = embedding_settings["collection_name"]

# Display settings
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Collection name:", COLLECTION_NAME)
print("Chroma path:", CHROMA_DIR)

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Collection name: sec_10k_rag_chunks
Chroma path: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\vectorstore\chroma_sec_10k


Load embedding model and Chroma collection

In [6]:
# Load the same embedding model used in notebook 05
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Create persistent Chroma client
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

# Load existing Chroma collection
collection = chroma_client.get_collection(
    name=COLLECTION_NAME
)

# Print collection status
print("Collection name:", COLLECTION_NAME)
print("Records in collection:", collection.count())
print("Rows in chunk dataset:", len(rag_chunks_df))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Collection name: sec_10k_rag_chunks
Records in collection: 520
Rows in chunk dataset: 520


#### Validate chroma

In [7]:
# Count records in Chroma
collection_count = collection.count()

# Count rows in the chunk dataset
chunk_count = len(rag_chunks_df)

# Print both counts
print("Chroma records:", collection_count)
print("Chunk rows:", chunk_count)

# Stop if the counts do not match
if collection_count != chunk_count:
    raise ValueError("Chroma collection count does not match RAG chunk dataset count.")

# Confirm validation passed
print("Chroma and chunk dataset match.")

Chroma records: 520
Chunk rows: 520
Chroma and chunk dataset match.


## Hybrid Retrieval Design

We will combine two search methods:

```text
Vector search
→ finds semantic meaning

BM25 keyword search
→ finds exact term matches
```

Then we will merge the two result lists using reciprocal rank fusion.

The idea:

```text
A chunk ranked highly by both methods should move higher.
```

Finally, we will rerank the merged candidates using a cross-encoder.

The full retrieval stack becomes:

```text
question
→ vector search top candidates
→ BM25 search top candidates
→ merge candidates
→ rerank candidates
→ final evidence chunks
```

#### Build Chroma filter helper

In [8]:
def build_chroma_where_filter(ticker=None, section_name=None):
    """
    Build a Chroma-compatible metadata filter.

    Chroma requires exactly one top-level filter condition.
    Multiple filters must be combined with $and.
    """

    # Create a list for filter conditions
    filter_conditions = []

    # Add ticker filter if provided
    if ticker is not None:
        filter_conditions.append({"ticker": ticker})

    # Add section filter if provided
    if section_name is not None:
        filter_conditions.append({"section_name": section_name})

    # Return no filter when no conditions exist
    if len(filter_conditions) == 0:
        return None

    # Return one filter directly
    if len(filter_conditions) == 1:
        return filter_conditions[0]

    # Combine multiple filters with Chroma's $and operator
    return {"$and": filter_conditions}

Vector search function

In [9]:
def vector_search(query, top_k=10, ticker=None, section_name=None):
    """
    Run semantic vector search against Chroma.
    """

    # Start timer
    start_time = time.time()

    # Embed the query
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    # Build metadata filter
    where_filter = build_chroma_where_filter(
        ticker=ticker,
        section_name=section_name
    )

    # Query Chroma
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where=where_filter,
        include=["documents", "metadatas", "distances"]
    )

    # End timer
    end_time = time.time()

    # Create result records
    result_records = []

    # Loop through results
    for rank, (doc, metadata, distance) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ),
        start=1
    ):

        # Store one vector search result
        result_records.append({
            "chunk_id": metadata.get("chunk_id", ""),
            "vector_rank": rank,
            "vector_distance": distance,
            "ticker": metadata.get("ticker", ""),
            "company_name": metadata.get("company_name", ""),
            "filing_date": metadata.get("filing_date", ""),
            "section_name": metadata.get("section_name", ""),
            "citation_label": metadata.get("citation_label", ""),
            "filing_url": metadata.get("filing_url", ""),
            "chunk_text": doc,
            "vector_time_seconds": round(end_time - start_time, 4)
        })

    # Return results as DataFrame
    return pd.DataFrame(result_records)

#### Tokenizer for BM25

In [10]:
def tokenize_text(text):
    """
    Convert text into simple lowercase tokens for BM25 keyword search.
    """

    # Convert text to string and lowercase it
    text = str(text).lower()

    # Keep only letters, numbers, and spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Split into tokens
    tokens = text.split()

    # Return tokens
    return tokens

BM25 search function

In [11]:
def bm25_search(query, top_k=10, ticker=None, section_name=None):
    """
    Run BM25 keyword search over the chunk dataset.

    This function supports optional ticker and section filters.
    """

    # Start timer
    start_time = time.time()

    # Create a filtered copy of the chunk dataset
    search_df = rag_chunks_df.copy()

    # Apply ticker filter if provided
    if ticker is not None:
        search_df = search_df[search_df["ticker"] == ticker].copy()

    # Apply section filter if provided
    if section_name is not None:
        search_df = search_df[search_df["section_name"] == section_name].copy()

    # Reset index after filtering
    search_df = search_df.reset_index(drop=True)

    # Return empty DataFrame if no records match filters
    if len(search_df) == 0:
        return pd.DataFrame()

    # Tokenize all chunk texts
    tokenized_corpus = [
        tokenize_text(text)
        for text in search_df["chunk_text"].tolist()
    ]

    # Build BM25 index
    bm25 = BM25Okapi(tokenized_corpus)

    # Tokenize the query
    tokenized_query = tokenize_text(query)

    # Get BM25 scores
    scores = bm25.get_scores(tokenized_query)

    # Add scores to search dataframe
    search_df["bm25_score"] = scores

    # Sort by BM25 score descending
    search_df = search_df.sort_values(
        "bm25_score",
        ascending=False
    ).head(top_k).copy()

    # Add BM25 rank
    search_df["bm25_rank"] = range(1, len(search_df) + 1)

    # End timer
    end_time = time.time()

    # Add timing
    search_df["bm25_time_seconds"] = round(end_time - start_time, 4)

    # Select output columns
    output_df = search_df[
        [
            "chunk_id",
            "bm25_rank",
            "bm25_score",
            "ticker",
            "company_name",
            "filing_date",
            "section_name",
            "citation_label",
            "filing_url",
            "chunk_text",
            "bm25_time_seconds"
        ]
    ].copy()

    # Return BM25 results
    return output_df

Test vector search and BM25 search

In [12]:
# Define a test query
test_query = "What supply chain risks does Tesla mention?"

# Run vector search
vector_results = vector_search(
    query=test_query,
    top_k=5,
    ticker="TSLA",
    section_name="item_1a_risk_factors"
)

# Run BM25 search
bm25_results = bm25_search(
    query=test_query,
    top_k=5,
    ticker="TSLA",
    section_name="item_1a_risk_factors"
)

# Display vector result summary
vector_results[
    [
        "vector_rank",
        "vector_distance",
        "ticker",
        "section_name",
        "citation_label"
    ]
]

,vector_rank,vector_distance,ticker,section_name,citation_label
0,1,0.867302,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
1,2,0.971211,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
2,3,1.006505,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
3,4,1.041513,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
4,5,1.047330,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."


Display BM25 result summary

In [14]:
# Display BM25 result summary
bm25_results[
    [
        "bm25_rank",
        "bm25_score",
        "ticker",
        "section_name",
        "citation_label"
    ]
]

,bm25_rank,bm25_score,ticker,section_name,citation_label
13,1,6.727838,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
20,2,3.073933,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
26,3,3.028184,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
22,4,2.907797,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
33,5,2.593929,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."


Hybrid search with reciprocal rank fusion

In [15]:
def hybrid_search(query, top_k=5, candidate_k=20, ticker=None, section_name=None, rrf_k=60):
    """
    Combine vector search and BM25 search using reciprocal rank fusion.

    RRF score formula:

    score = 1 / (rrf_k + rank)

    A chunk gets points from vector search, BM25 search, or both.
    """

    # Run vector search with more candidates than final top_k
    vector_df = vector_search(
        query=query,
        top_k=candidate_k,
        ticker=ticker,
        section_name=section_name
    )

    # Run BM25 search with more candidates than final top_k
    bm25_df = bm25_search(
        query=query,
        top_k=candidate_k,
        ticker=ticker,
        section_name=section_name
    )

    # Create dictionary to store merged records
    merged_records = {}

    # Add vector search results to merged records
    for _, row in vector_df.iterrows():

        # Get chunk ID
        chunk_id = row["chunk_id"]

        # Initialize record if this chunk has not been seen
        if chunk_id not in merged_records:
            merged_records[chunk_id] = {
                "chunk_id": chunk_id,
                "ticker": row["ticker"],
                "company_name": row["company_name"],
                "filing_date": row["filing_date"],
                "section_name": row["section_name"],
                "citation_label": row["citation_label"],
                "filing_url": row["filing_url"],
                "chunk_text": row["chunk_text"],
                "vector_rank": None,
                "vector_distance": None,
                "bm25_rank": None,
                "bm25_score": None,
                "rrf_score": 0
            }

        # Add vector rank and distance
        merged_records[chunk_id]["vector_rank"] = row["vector_rank"]
        merged_records[chunk_id]["vector_distance"] = row["vector_distance"]

        # Add RRF score from vector rank
        merged_records[chunk_id]["rrf_score"] += 1 / (rrf_k + row["vector_rank"])

    # Add BM25 results to merged records
    for _, row in bm25_df.iterrows():

        # Get chunk ID
        chunk_id = row["chunk_id"]

        # Initialize record if this chunk has not been seen
        if chunk_id not in merged_records:
            merged_records[chunk_id] = {
                "chunk_id": chunk_id,
                "ticker": row["ticker"],
                "company_name": row["company_name"],
                "filing_date": row["filing_date"],
                "section_name": row["section_name"],
                "citation_label": row["citation_label"],
                "filing_url": row["filing_url"],
                "chunk_text": row["chunk_text"],
                "vector_rank": None,
                "vector_distance": None,
                "bm25_rank": None,
                "bm25_score": None,
                "rrf_score": 0
            }

        # Add BM25 rank and score
        merged_records[chunk_id]["bm25_rank"] = row["bm25_rank"]
        merged_records[chunk_id]["bm25_score"] = row["bm25_score"]

        # Add RRF score from BM25 rank
        merged_records[chunk_id]["rrf_score"] += 1 / (rrf_k + row["bm25_rank"])

    # Convert merged records into DataFrame
    hybrid_df = pd.DataFrame(list(merged_records.values()))

    # Sort by RRF score
    hybrid_df = hybrid_df.sort_values(
        "rrf_score",
        ascending=False
    ).head(top_k).reset_index(drop=True)

    # Add final hybrid rank
    hybrid_df["hybrid_rank"] = range(1, len(hybrid_df) + 1)

    # Reorder columns
    hybrid_df = hybrid_df[
        [
            "hybrid_rank",
            "rrf_score",
            "chunk_id",
            "vector_rank",
            "vector_distance",
            "bm25_rank",
            "bm25_score",
            "ticker",
            "company_name",
            "filing_date",
            "section_name",
            "citation_label",
            "filing_url",
            "chunk_text"
        ]
    ]

    # Return hybrid results
    return hybrid_df

Test hybrid search

In [16]:
# Run hybrid search
hybrid_results = hybrid_search(
    query=test_query,
    top_k=5,
    candidate_k=20,
    ticker="TSLA",
    section_name="item_1a_risk_factors"
)

# Display hybrid result summary
hybrid_results[
    [
        "hybrid_rank",
        "rrf_score",
        "vector_rank",
        "bm25_rank",
        "ticker",
        "section_name",
        "citation_label"
    ]
]

,hybrid_rank,rrf_score,vector_rank,bm25_rank,ticker,section_name,citation_label
0,1,0.032002,2.0,3.0,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
1,2,0.031281,6.0,2.0,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
2,3,0.030366,3.0,9.0,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
3,4,0.030331,4.0,8.0,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
4,5,0.030310,5.0,7.0,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."


## Reranking

Hybrid search gives us a strong candidate set.

Now we add a cross-encoder reranker.

A cross-encoder looks at:

```text
question + candidate chunk
```

Then it gives a relevance score.

This is usually more accurate than embeddings alone because the model compares the query and chunk together.

The tradeoff:

```text
better ranking
but slower retrieval
```

In [17]:
# Set reranker model name
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Start timer
start_time = time.time()

# Load cross-encoder reranker
reranker_model = CrossEncoder(RERANKER_MODEL_NAME)

# End timer
end_time = time.time()

# Print reranker loading time
print("Reranker model:", RERANKER_MODEL_NAME)
print("Loaded in seconds:", round(end_time - start_time, 2))

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\.venv\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tevin\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2
Loaded in seconds: 6.78


Rerank hybrid results

In [18]:
def rerank_results(query, candidate_df, top_k=5):
    """
    Rerank candidate chunks using a cross-encoder.

    The cross-encoder reads query and chunk text together.
    """

    # Return empty DataFrame if there are no candidates
    if candidate_df.empty:
        return pd.DataFrame()

    # Create query-document pairs for reranking
    pairs = [
        [query, chunk_text]
        for chunk_text in candidate_df["chunk_text"].tolist()
    ]

    # Start timer
    start_time = time.time()

    # Predict relevance scores
    rerank_scores = reranker_model.predict(pairs)

    # End timer
    end_time = time.time()

    # Copy candidate results
    reranked_df = candidate_df.copy()

    # Add reranker scores
    reranked_df["rerank_score"] = rerank_scores

    # Sort by reranker score descending
    reranked_df = reranked_df.sort_values(
        "rerank_score",
        ascending=False
    ).head(top_k).reset_index(drop=True)

    # Add rerank rank
    reranked_df["rerank_rank"] = range(1, len(reranked_df) + 1)

    # Add rerank time
    reranked_df["rerank_time_seconds"] = round(end_time - start_time, 4)

    # Reorder columns
    reranked_df = reranked_df[
        [
            "rerank_rank",
            "rerank_score",
            "hybrid_rank",
            "rrf_score",
            "chunk_id",
            "vector_rank",
            "bm25_rank",
            "ticker",
            "company_name",
            "filing_date",
            "section_name",
            "citation_label",
            "filing_url",
            "chunk_text",
            "rerank_time_seconds"
        ]
    ]

    # Return reranked results
    return reranked_df

#### Test reranking

In [19]:
# Create a larger hybrid candidate set for reranking
hybrid_candidates = hybrid_search(
    query=test_query,
    top_k=15,
    candidate_k=30,
    ticker="TSLA",
    section_name="item_1a_risk_factors"
)

# Rerank the hybrid candidates
reranked_results = rerank_results(
    query=test_query,
    candidate_df=hybrid_candidates,
    top_k=5
)

# Display reranked summary
reranked_results[
    [
        "rerank_rank",
        "rerank_score",
        "hybrid_rank",
        "vector_rank",
        "bm25_rank",
        "ticker",
        "section_name",
        "citation_label"
    ]
]

,rerank_rank,rerank_score,hybrid_rank,vector_rank,bm25_rank,ticker,section_name,citation_label
0,1,0.382475,1,2.0,3.0,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
1,2,-0.133744,3,3.0,9.0,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
2,3,-0.744060,13,12.0,20.0,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
3,4,-0.824206,5,5.0,7.0,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
4,5,-2.320509,7,13.0,6.0,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."


Preview reranked evidence

In [20]:
def preview_retrieved_evidence(results_df, text_chars=700):
    """
    Print retrieved evidence in a readable format.
    """

    # Handle empty results
    if results_df.empty:
        print("No results found.")
        return

    # Loop through results
    for _, row in results_df.iterrows():

        # Print separator
        print("=" * 100)

        # Print rank information
        if "rerank_rank" in row:
            print("Rerank rank:", row["rerank_rank"])
            print("Rerank score:", row["rerank_score"])
        elif "hybrid_rank" in row:
            print("Hybrid rank:", row["hybrid_rank"])

        # Print citation metadata
        print("Ticker:", row["ticker"])
        print("Company:", row["company_name"])
        print("Section:", row["section_name"])
        print("Citation:", row["citation_label"])
        print("Filing URL:", row["filing_url"])
        print("-" * 100)

        # Print wrapped chunk text
        preview_text = str(row["chunk_text"])[:text_chars]
        print(textwrap.fill(preview_text, width=110))
        print()

Preview reranked results

In [21]:
# Preview the final reranked evidence
preview_retrieved_evidence(
    results_df=reranked_results,
    text_chars=900
)

Rerank rank: 1
Rerank score: 0.38247478008270264
Ticker: TSLA
Company: Tesla
Section: item_1a_risk_factors
Citation: TSLA 2026-01-29 10-K, item_1a_risk_factors, chunk 26
Filing URL: https://www.sec.gov/Archives/edgar/data/1318605/000162828026003952/tsla-20251231.htm
----------------------------------------------------------------------------------------------------
a regulator to contain a safety defect or be noncompliant with applicable laws and regulations, such as U.S.
Federal Motor Vehicle Safety Standards, or provide certain functionalities. Such recalls or field actions,
whether voluntary or involuntary or caused by systems or components engineered or manufactured by us or our
suppliers, could result in significant expense, supply chain complications and service burdens, and may harm
our brand, business, prospects, financial condition and operating results. Our current and future warranty
reserves may be insufficient to cover future warranty claims. We provide a manufacturer’s wa

## Retrieval Method Comparison

Now we compare:

```text
vector search only
BM25 keyword search only
hybrid search
hybrid search + reranking
```

This helps show why each retrieval layer matters.

#### Compare retrieval methods for one query

In [22]:
# Define comparison query
comparison_query = "What cybersecurity risks does Microsoft mention?"

# Vector-only results
comparison_vector = vector_search(
    query=comparison_query,
    top_k=5,
    ticker="MSFT",
    section_name="item_1a_risk_factors"
)

# BM25-only results
comparison_bm25 = bm25_search(
    query=comparison_query,
    top_k=5,
    ticker="MSFT",
    section_name="item_1a_risk_factors"
)

# Hybrid results
comparison_hybrid = hybrid_search(
    query=comparison_query,
    top_k=10,
    candidate_k=30,
    ticker="MSFT",
    section_name="item_1a_risk_factors"
)

# Reranked hybrid results
comparison_reranked = rerank_results(
    query=comparison_query,
    candidate_df=comparison_hybrid,
    top_k=5
)

# Create comparison table
method_comparison = pd.DataFrame([
    {
        "method": "vector",
        "top_citation": comparison_vector.iloc[0]["citation_label"],
        "top_chunk_preview": comparison_vector.iloc[0]["chunk_text"][:300]
    },
    {
        "method": "bm25",
        "top_citation": comparison_bm25.iloc[0]["citation_label"],
        "top_chunk_preview": comparison_bm25.iloc[0]["chunk_text"][:300]
    },
    {
        "method": "hybrid",
        "top_citation": comparison_hybrid.iloc[0]["citation_label"],
        "top_chunk_preview": comparison_hybrid.iloc[0]["chunk_text"][:300]
    },
    {
        "method": "hybrid_plus_rerank",
        "top_citation": comparison_reranked.iloc[0]["citation_label"],
        "top_chunk_preview": comparison_reranked.iloc[0]["chunk_text"][:300]
    }
])

# Display comparison
method_comparison

,method,top_citation,top_chunk_preview
0,vector,"MSFT 2025-07-30 10-K, item_1a_risk_factors, ch...",previously disclosed in our Form 8-K filed wit...
1,bm25,"MSFT 2025-07-30 10-K, item_1a_risk_factors, ch...",cause an impairment of goodwill or intangibles...
2,hybrid,"MSFT 2025-07-30 10-K, item_1a_risk_factors, ch...",cause an impairment of goodwill or intangibles...
3,hybrid_plus_rerank,"MSFT 2025-07-30 10-K, item_1a_risk_factors, ch...",cause an impairment of goodwill or intangibles...


Create reusable final retriever

In [23]:
def retrieve_evidence(query, top_k=5, candidate_k=30, ticker=None, section_name=None, use_reranker=True):
    """
    Final evidence retriever for RiskRadar AI.

    Steps:
    1. Run hybrid search.
    2. Optionally rerank the hybrid candidates.
    3. Return final citation-ready evidence.
    """

    # Run hybrid search to get candidate chunks
    hybrid_candidates = hybrid_search(
        query=query,
        top_k=candidate_k,
        candidate_k=candidate_k,
        ticker=ticker,
        section_name=section_name
    )

    # Return empty DataFrame if no candidates are found
    if hybrid_candidates.empty:
        return pd.DataFrame()

    # Use reranker if requested
    if use_reranker:
        final_results = rerank_results(
            query=query,
            candidate_df=hybrid_candidates,
            top_k=top_k
        )

    # Otherwise return top hybrid results
    else:
        final_results = hybrid_candidates.head(top_k).copy()

    # Return final evidence
    return final_results

Test final retriever

In [24]:
# Define final test query
final_query = "What AI and competition risks does NVIDIA mention?"

# Retrieve final evidence
final_evidence = retrieve_evidence(
    query=final_query,
    top_k=5,
    candidate_k=30,
    ticker="NVDA",
    section_name="item_1a_risk_factors",
    use_reranker=True
)

# Display compact final evidence table
final_evidence[
    [
        "rerank_rank",
        "rerank_score",
        "ticker",
        "section_name",
        "citation_label"
    ]
]

,rerank_rank,rerank_score,ticker,section_name,citation_label
0,1,1.623981,NVDA,item_1a_risk_factors,"NVDA 2026-02-25 10-K, item_1a_risk_factors, ch..."
1,2,1.410849,NVDA,item_1a_risk_factors,"NVDA 2026-02-25 10-K, item_1a_risk_factors, ch..."
2,3,0.331172,NVDA,item_1a_risk_factors,"NVDA 2026-02-25 10-K, item_1a_risk_factors, ch..."
3,4,-2.061538,NVDA,item_1a_risk_factors,"NVDA 2026-02-25 10-K, item_1a_risk_factors, ch..."
4,5,-2.639408,NVDA,item_1a_risk_factors,"NVDA 2026-02-25 10-K, item_1a_risk_factors, ch..."


Test final retriever

In [26]:
# Preview final retrieved evidence
preview_retrieved_evidence(
    results_df=final_evidence,
    text_chars=900
)

Rerank rank: 1
Rerank score: 1.623981237411499
Ticker: NVDA
Company: NVIDIA
Section: item_1a_risk_factors
Citation: NVDA 2026-02-25 10-K, item_1a_risk_factors, chunk 41
Filing URL: https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm
----------------------------------------------------------------------------------------------------
for information from competition regulators in the European Union, the United States, the United Kingdom,
China, and South Korea regarding our sales of GPUs and other NVIDIA products, our efforts to allocate supply,
foundation models and our investments, partnerships and other agreements with companies developing foundation
models, the markets in which we compete and our competition, our strategies, roadmaps, and efforts to develop,
market, and sell hardware, software, and system solutions, and our agreements with customers, suppliers, and
partners. We expect to receive additional requests for information in the future. Such 

In [27]:
# Create output paths
hybrid_results_file = PROCESSED_DIR / "sec_10k_sample_hybrid_results.csv"
reranked_results_file = PROCESSED_DIR / "sec_10k_sample_reranked_results.csv"
method_comparison_file = PROCESSED_DIR / "sec_10k_retrieval_method_comparison.csv"

# Save sample hybrid results
hybrid_results.to_csv(hybrid_results_file, index=False)

# Save sample reranked results
reranked_results.to_csv(reranked_results_file, index=False)

# Save method comparison
method_comparison.to_csv(method_comparison_file, index=False)

# Confirm files were saved
print("Saved hybrid results to:", hybrid_results_file)
print("Saved reranked results to:", reranked_results_file)
print("Saved method comparison to:", method_comparison_file)

Saved hybrid results to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_sample_hybrid_results.csv
Saved reranked results to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_sample_reranked_results.csv
Saved method comparison to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_retrieval_method_comparison.csv


In [28]:
# Create final checkpoint table
hybrid_checkpoint = pd.DataFrame({
    "output": [
        "Sample hybrid retrieval results",
        "Sample reranked retrieval results",
        "Retrieval method comparison"
    ],
    "path": [
        str(hybrid_results_file),
        str(reranked_results_file),
        str(method_comparison_file)
    ],
    "exists": [
        hybrid_results_file.exists(),
        reranked_results_file.exists(),
        method_comparison_file.exists()
    ]
})

# Display checkpoint table
hybrid_checkpoint

,output,path,exists
0,Sample hybrid retrieval results,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
1,Sample reranked retrieval results,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
2,Retrieval method comparison,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True


In [29]:
# Validate that notebook 07 completed successfully

# Check that the main output files exist
print("Hybrid results file exists:", hybrid_results_file.exists())
print("Reranked results file exists:", reranked_results_file.exists())
print("Method comparison file exists:", method_comparison_file.exists())

# Check that final evidence retrieval returned results
print("Final evidence rows:", len(final_evidence))

# Check that reranker output has the expected columns
expected_final_columns = [
    "rerank_rank",
    "rerank_score",
    "ticker",
    "section_name",
    "citation_label",
    "filing_url",
    "chunk_text"
]

# Find missing columns
missing_final_columns = [
    column for column in expected_final_columns
    if column not in final_evidence.columns
]

# Stop if important columns are missing
if missing_final_columns:
    raise ValueError(f"Missing columns in final evidence: {missing_final_columns}")

# Confirm notebook completed successfully
print("Notebook 07 completed successfully.")

Hybrid results file exists: True
Reranked results file exists: True
Method comparison file exists: True
Final evidence rows: 5
Notebook 07 completed successfully.


## Hybrid Search and Reranking Conclusion

This notebook improved the retrieval layer of RiskRadar AI.

The project now has:

```text
semantic vector search
+ BM25 keyword search
+ reciprocal rank fusion
+ cross-encoder reranking
```

This makes the RAG system stronger because it can retrieve evidence using both meaning and exact keywords.

The final retriever can now return citation-ready SEC evidence for a question.

The next notebook will use this evidence to generate grounded answers.

```text
question
→ retrieve evidence
→ build prompt
→ generate answer
→ return citations
```